# TVMN/NUVM Submission-Ready Results Notebook

This notebook collects the actual numerical outputs needed for a submission-ready TVMN/NUVM distribution paper:

- parameter estimates tables
- log-likelihood, AIC, BIC and related model-comparison tables
- KS, Anderson-Darling, and Cramer-von Mises diagnostics
- bootstrap confidence intervals
- simulation results
- MLE recovery tables
- final PDF/CDF equations for NUVM and TVMN

Run the notebook from `C:/Users/LENOVO/Documents/distribution`.

In [2]:
from pathlib import Path
import importlib.util
import sys

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display, Markdown

BASE = Path.cwd()
TVMN_DIR = BASE / 'TVMN'
NUVM_DIR = BASE / 'NUVM'
TVMN_OUT = TVMN_DIR / 'outputs'
NUVM_OUT = NUVM_DIR

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)
pd.set_option('display.precision', 6)

def read_csv(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

def show(title, df, n=None):
    display(Markdown(f'## {title}'))
    display(df if n is None else df.head(n))

def load_module(module_name, path):
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

tvmn_deriv = load_module('tvmn_deriv_notebook', TVMN_DIR / '01_TVMN_step_by_step_derivation.py')
nuvm_mle = load_module('nuvm_mle_notebook', NUVM_DIR / 'NUVM_MLE.py')

def nuvm_cdf(x, mu, sigma, alpha, quadrature_points=80):
    x = np.asarray(x, dtype=float)
    scalar_input = x.ndim == 0
    x = np.atleast_1d(x)
    lower_v = (1.0 - alpha) * sigma * sigma
    upper_v = (1.0 + alpha) * sigma * sigma
    nodes, weights = np.polynomial.legendre.leggauss(quadrature_points)
    v = lower_v + 0.5 * (nodes + 1.0) * (upper_v - lower_v)
    w = 0.5 * weights
    cdf = np.sum(w[:, None] * stats.norm.cdf((x[None, :] - mu) / np.sqrt(v[:, None])), axis=0)
    return float(cdf[0]) if scalar_input else np.clip(cdf, 0.0, 1.0)

def empirical_ks(data, cdf_func):
    x = np.sort(np.asarray(data, dtype=float))
    n = len(x)
    cdf_values = np.clip(cdf_func(x), 0.0, 1.0)
    d_plus = np.max(np.arange(1, n + 1) / n - cdf_values)
    d_minus = np.max(cdf_values - np.arange(0, n) / n)
    return float(max(d_plus, d_minus))

print('Base directory:', BASE)
print('TVMN outputs:', TVMN_OUT)
print('NUVM outputs:', NUVM_OUT)

Base directory: c:\Users\LENOVO\Documents\distribution
TVMN outputs: c:\Users\LENOVO\Documents\distribution\TVMN\outputs
NUVM outputs: c:\Users\LENOVO\Documents\distribution\NUVM


# Final Model Equations

## NUVM distribution

Let

$$V \sim U(v_L,v_U), \qquad v_L=(1-\alpha)\sigma^2, \qquad v_U=(1+\alpha)\sigma^2,$$

where $\sigma>0$ and $0<\alpha<1$. Conditional on $V=v$,

$$X\mid V=v \sim N(\mu,v).$$

The NUVM PDF is

$$f_{NUVM}(x;\mu,\sigma,\alpha)=\frac{1}{v_U-v_L}\int_{v_L}^{v_U}\frac{1}{\sqrt{2\pi v}}\exp\left[-\frac{(x-\mu)^2}{2v}\right]dv.$$

With $z=|x-\mu|$ and

$$A(v,z)=2\sqrt{v}\exp\left(-\frac{z^2}{2v}\right)+\sqrt{2\pi}z\,\operatorname{erf}\left(\frac{z}{\sqrt{2v}}\right),$$

the implemented closed-form PDF is

$$f_{NUVM}(x;\mu,\sigma,\alpha)=\frac{A(v_U,z)-A(v_L,z)}{(v_U-v_L)\sqrt{2\pi}}.$$

The NUVM CDF is evaluated as

$$F_{NUVM}(x;\mu,\sigma,\alpha)=\frac{1}{v_U-v_L}\int_{v_L}^{v_U}\Phi\left(\frac{x-\mu}{\sqrt v}\right)dv.$$

## TVMN distribution

Let

$$V\sim Triangular(a,m,b), \qquad 0<a<m<b,$$

with triangular variance density

$$g(v)=\begin{cases}
\dfrac{2(v-a)}{(b-a)(m-a)}, & a\le v\le m,\\[6pt]
\dfrac{2(b-v)}{(b-a)(b-m)}, & m<v\le b,\\[6pt]
0, & \text{otherwise.}
\end{cases}$$

Conditional on $V=v$,

$$X\mid V=v\sim N(0,v).$$

The TVMN PDF is

$$f_{TVMN}(x;a,m,b)=\int_a^b \frac{1}{\sqrt{2\pi v}}\exp\left(-\frac{x^2}{2v}\right)g(v)dv.$$

Let $\lambda=x^2/2$ and define

$$A(v)=2\sqrt v\exp(-\lambda/v)-2\sqrt{\pi\lambda}\operatorname{erfc}\left(\sqrt{\lambda/v}\right),$$

$$B(v)=\frac{2}{3}v^{3/2}\exp(-\lambda/v)-\frac{4}{3}\lambda\sqrt v\exp(-\lambda/v)+\frac{4}{3}\sqrt\pi\lambda^{3/2}\operatorname{erfc}\left(\sqrt{\lambda/v}\right).$$

Then the implemented TVMN PDF is

$$f_{TVMN}(x)=\frac{2}{(b-a)\sqrt{2\pi}}\left[\frac{B(m)-B(a)-a\{A(m)-A(a)\}}{m-a}+\frac{b\{A(b)-A(m)\}-\{B(b)-B(m)\}}{b-m}\right].$$

The TVMN CDF is evaluated as

$$F_{TVMN}(x;a,m,b)=\int_a^b \Phi\left(\frac{x}{\sqrt v}\right)g(v)dv,$$

with numerical quadrature split at $m$. For raw-data benchmark fitting, the notebook uses the location extension $Y=\mu+X$.

# Parameter Estimates Tables

In [3]:
show('TVMN real-data parameter estimates', read_csv(TVMN_OUT / '04_TVMN_real_data_parameter_estimates.csv'))
show('TVMN expanded benchmark parameter estimates', read_csv(TVMN_OUT / '05_TVMN_benchmark_parameter_estimates.csv'))
show('NUVM real-data parameter estimates', read_csv(NUVM_OUT / '05_NUVM_real_data_parameter_estimates.csv'))

## TVMN real-data parameter estimates

,dataset,model,mu,sigma,loc,scale,alpha,a,m,b
0,diabetes_target_centered,Normal,-7.716301e-15,77.005746,NaN,NaN,NaN,NaN,NaN,NaN
1,diabetes_target_centered,Laplace,NaN,NaN,-11.633484,65.042986,NaN,NaN,NaN,NaN
2,diabetes_target_centered,NUVM,6.587163e-04,77.006532,NaN,NaN,0.000100,NaN,NaN,NaN
3,diabetes_target_centered,TVMN,NaN,NaN,NaN,NaN,NaN,5828.537585,5894.920113,6060.133876
4,breast_cancer_mean_radius_centered,Normal,-4.995028e-16,3.520951,NaN,NaN,NaN,NaN,NaN,NaN
5,breast_cancer_mean_radius_centered,Laplace,NaN,NaN,-0.757292,2.665731,NaN,NaN,NaN,NaN
6,breast_cancer_mean_radius_centered,NUVM,-4.851936e-01,3.541842,NaN,NaN,0.943982,NaN,NaN,NaN
7,breast_cancer_mean_radius_centered,TVMN,NaN,NaN,NaN,NaN,NaN,5.065086,5.163014,26.849543
8,breast_cancer_mean_area_centered,Normal,-6.713318e-14,351.604754,NaN,NaN,NaN,NaN,NaN,NaN
9,breast_cancer_mean_area_centered,Laplace,NaN,NaN,-103.789104,246.513533,NaN,NaN,NaN,NaN


## TVMN expanded benchmark parameter estimates

,dataset,model,mu,sigma,loc,scale,df,shape,alpha,a,m,b
0,diabetes_target,Normal,152.133484,77.005746,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,diabetes_target,Laplace,NaN,NaN,140.50000,65.042986,NaN,NaN,NaN,NaN,NaN,NaN
2,diabetes_target,StudentT,NaN,NaN,152.12481,76.985218,127967.505294,NaN,NaN,NaN,NaN,NaN
3,diabetes_target,Gamma,NaN,NaN,0.00000,41.747229,NaN,3.644158,NaN,NaN,NaN,NaN
4,diabetes_target,Lognormal,NaN,NaN,0.00000,131.804917,NaN,0.557978,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
67,linnerud_weight,Lognormal,NaN,NaN,0.00000,35.273528,NaN,0.082999,NaN,NaN,NaN,NaN
68,linnerud_weight,Weibull,NaN,NaN,0.00000,36.945765,NaN,9.199607,NaN,NaN,NaN,NaN
69,linnerud_weight,Exponential,NaN,NaN,0.00000,35.400000,NaN,NaN,NaN,NaN,NaN,NaN
70,linnerud_weight,NUVM,35.163432,3.030218,NaN,NaN,NaN,NaN,0.876297,NaN,NaN,NaN


## NUVM real-data parameter estimates

,Model,mu,sigma,a/df
0,Normal,0.042126,1.054237,NaN
1,Student-t,0.080089,0.680329,3.568405
2,NUVM,0.108308,1.246890,0.999000


# AIC/BIC Model-Comparison Tables

In [4]:
tvmn_benchmark = read_csv(TVMN_OUT / '05_TVMN_benchmark_model_comparison.csv')
show('TVMN expanded benchmark: LogLik, AIC, BIC, KS', tvmn_benchmark[['dataset','n','model','loglik','k','AIC','BIC','CAIC','HQIC','KS']].sort_values(['dataset','AIC']))
show('TVMN benchmark AIC/BIC winners', read_csv(TVMN_OUT / '05_TVMN_benchmark_winners.csv'))
show('TVMN versus NUVM benchmark comparison', read_csv(TVMN_OUT / '05_TVMN_benchmark_TVMN_vs_NUVM.csv'))
show('NUVM model-comparison table', read_csv(NUVM_OUT / '05_NUVM_real_data_model_comparison.csv'))
show('NUVM multi-dataset comparison', read_csv(NUVM_OUT / 'NUVM_multi_dataset_comparison.csv'))

## TVMN expanded benchmark: LogLik, AIC, BIC, KS

,dataset,n,model,loglik,k,AIC,BIC,CAIC,HQIC,KS
31,breast_cancer_mean_area,569,Lognormal,-4013.608565,2,8031.217130,8039.904891,8041.904891,8034.607093,0.063260
30,breast_cancer_mean_area,569,Gamma,-4036.295384,2,8076.590768,8085.278529,8087.278529,8079.980731,0.097473
32,breast_cancer_mean_area,569,Weibull,-4077.388996,2,8158.777992,8167.465753,8169.465753,8162.167955,0.117941
28,breast_cancer_mean_area,569,Laplace,-4097.120956,2,8198.241913,8206.929674,8208.929674,8201.631876,0.120606
35,breast_cancer_mean_area,569,TVMN,-4106.342318,4,8220.684635,8238.060157,8242.060157,8227.464560,0.124515
...,...,...,...,...,...,...,...,...,...,...
61,wine_proline,178,NUVM,-1275.975288,3,2557.950575,2567.495926,2570.495926,2561.821471,0.129320
56,wine_proline,178,StudentT,-1275.975634,3,2557.951268,2567.496619,2570.496619,2561.822164,0.129282
62,wine_proline,178,TVMN,-1275.976557,4,2559.953114,2572.680249,2576.680249,2565.114309,0.128346
55,wine_proline,178,Laplace,-1283.811818,2,2571.623636,2577.987203,2579.987203,2574.204233,0.114991


## TVMN benchmark AIC/BIC winners

,dataset,n,criterion,winner,winner_value,tvmn_value,delta_tvmn_minus_best
0,diabetes_target,442,AIC,Gamma,5042.102931,5102.342903,60.239972
1,diabetes_target,442,BIC,Gamma,5050.285550,5118.708142,68.422592
2,diabetes_target,442,CAIC,Gamma,5052.285550,5122.708142,70.422592
3,diabetes_target,442,HQIC,Gamma,5045.330383,5108.797808,63.467425
4,breast_cancer_mean_radius,569,AIC,Lognormal,2965.641767,3041.637825,75.996058
5,breast_cancer_mean_radius,569,BIC,Lognormal,2974.329528,3059.013347,84.683819
6,breast_cancer_mean_radius,569,CAIC,Lognormal,2976.329528,3063.013347,86.683819
7,breast_cancer_mean_radius,569,HQIC,Lognormal,2969.031729,3048.417750,79.386021
8,breast_cancer_mean_texture,569,AIC,Lognormal,3239.084775,3276.999010,37.914235
9,breast_cancer_mean_texture,569,BIC,Lognormal,3247.772536,3294.374532,46.601996


## TVMN versus NUVM benchmark comparison

,dataset,n,tvmn_loglik,nuvm_loglik,loglik_tvmn_minus_nuvm,delta_aic_tvmn_minus_nuvm,delta_bic_tvmn_minus_nuvm,tvmn_ks,nuvm_ks,delta_ks_tvmn_minus_nuvm
0,diabetes_target,442,-2547.171451,-2547.165810,-0.005641,2.011282,6.102592,0.096166,0.096039,0.000127
1,breast_cancer_mean_radius,569,-1516.818912,-1517.357330,0.538418,0.923164,5.267045,0.071900,0.066383,0.005518
2,breast_cancer_mean_texture,569,-1634.499505,-1634.900527,0.401022,1.197957,5.541837,0.037387,0.038416,-0.001029
3,breast_cancer_mean_area,569,-4106.342318,-4114.481199,8.138882,-14.277763,-9.933883,0.124515,0.125356,-0.000841
4,wine_alcohol,178,-214.962779,-214.962241,-0.000538,2.001077,5.182860,0.068165,0.068541,-0.000376
5,wine_malic_acid,178,-271.360892,-271.469657,0.108765,1.782471,4.964254,0.169421,0.174164,-0.004743
6,wine_proline,178,-1275.976557,-1275.975288,-0.001270,2.002539,5.184323,0.128346,0.129320,-0.000974
7,linnerud_weight,20,-49.561554,-49.896233,0.334679,1.330643,2.326375,0.118080,0.119535,-0.001455


## NUVM model-comparison table

,Model,k,LogLik,AIC,BIC
0,Normal,2,-3617.574670,7239.149340,7250.763547
1,Student-t,3,-3268.376091,6542.752181,6560.173491
2,NUVM,3,-3475.823497,6957.646993,6975.068303


## NUVM multi-dataset comparison

,Dataset,Ticker,n,SampleKurtosis,SampleSkewness,Model,mu,sigma,shape,LogLik,AIC,BIC
0,NIFTY,^NSEI,2458,23.369296,-1.406784,Normal,0.042126,1.054237,NaN,-3617.574670,7239.149340,7250.763547
1,NIFTY,^NSEI,2458,23.369296,-1.406784,Student-t,0.080089,0.680329,3.568405,-3268.376091,6542.752181,6560.173491
2,NIFTY,^NSEI,2458,23.369296,-1.406784,NUVM-MLE,0.116725,1.269138,0.957607,-3498.620933,7003.241867,7020.663177
3,NIFTY,^NSEI,2458,23.369296,-1.406784,NUVM-MoM,0.042126,1.054237,0.999000,-4053.544999,NaN,NaN
4,SP500,^GSPC,2515,18.721439,-0.809953,Normal,0.041750,1.126748,NaN,-3868.760160,7741.520320,7753.180376
5,SP500,^GSPC,2515,18.721439,-0.809953,Student-t,0.081666,0.642740,2.680564,-3459.715051,6925.430102,6942.920186
6,SP500,^GSPC,2515,18.721439,-0.809953,NUVM-MLE,0.043811,1.161577,0.999000,-3613.853322,7233.706644,7251.196728
7,SP500,^GSPC,2515,18.721439,-0.809953,NUVM-MoM,0.041750,1.126748,0.999000,-4261.206231,NaN,NaN
8,NASDAQ,^IXIC,2515,11.018173,-0.640308,Normal,0.055961,1.348293,NaN,-4320.211380,8644.422759,8656.082816
9,NASDAQ,^IXIC,2515,11.018173,-0.640308,Student-t,0.123146,0.855757,3.011099,-4065.123408,8136.246816,8153.736901


# KS / Anderson-Darling / Cramer-von Mises Diagnostics

The TVMN benchmark CSV already contains KS. The following cell computes AD and CVM from the fitted CDF values for TVMN and NUVM on the benchmark datasets.

In [5]:
def ad_cvm_from_cdf_values(u):
    u = np.sort(np.clip(np.asarray(u, dtype=float), 1e-10, 1 - 1e-10))
    n = len(u)
    i = np.arange(1, n + 1)
    cvm = 1.0 / (12.0 * n) + np.sum((u - (2.0 * i - 1.0) / (2.0 * n)) ** 2)
    ad = -n - np.mean((2.0 * i - 1.0) * (np.log(u) + np.log(1.0 - u[::-1])))
    return float(ad), float(cvm)

saved_diag_path = TVMN_OUT / '05_TVMN_benchmark_KS_AD_CVM.csv'

try:
    from sklearn import datasets

    def load_benchmark_datasets_local():
        loaded = []
        diabetes = datasets.load_diabetes()
        loaded.append(('diabetes_target', diabetes.target))
        breast = datasets.load_breast_cancer()
        loaded.append(('breast_cancer_mean_radius', breast.data[:, 0]))
        loaded.append(('breast_cancer_mean_texture', breast.data[:, 1]))
        loaded.append(('breast_cancer_mean_area', breast.data[:, 3]))
        wine = datasets.load_wine()
        loaded.append(('wine_alcohol', wine.data[:, 0]))
        loaded.append(('wine_malic_acid', wine.data[:, 1]))
        loaded.append(('wine_proline', wine.data[:, 12]))
        linnerud = datasets.load_linnerud()
        loaded.append(('linnerud_weight', linnerud.target[:, 1]))
        clean = []
        for name, values in loaded:
            values = np.asarray(values, dtype=float)
            values = values[np.isfinite(values)]
            values = values[values > 0.0]
            if len(values) >= 20 and np.std(values) > 0:
                clean.append((name, values))
        return clean

    params = read_csv(TVMN_OUT / '05_TVMN_benchmark_parameter_estimates.csv')
    diag_rows = []
    for dataset_name, data in load_benchmark_datasets_local():
        data = np.asarray(data, dtype=float)
        for model_name in ['TVMN', 'NUVM']:
            row = params[(params['dataset'] == dataset_name) & (params['model'] == model_name)].iloc[0]
            if model_name == 'TVMN':
                cdf_func = lambda x, row=row: tvmn_deriv.tvmn_cdf(np.asarray(x) - row['mu'], row['a'], row['m'], row['b'])
            else:
                cdf_func = lambda x, row=row: nuvm_cdf(x, row['mu'], row['sigma'], row['alpha'])
            u = cdf_func(data)
            ks = empirical_ks(data, cdf_func)
            ad, cvm = ad_cvm_from_cdf_values(u)
            diag_rows.append({'dataset': dataset_name, 'n': len(data), 'model': model_name, 'KS': ks, 'AD': ad, 'CVM': cvm})
    tvmn_nuvm_ad_cvm = pd.DataFrame(diag_rows)
    tvmn_nuvm_ad_cvm.to_csv(saved_diag_path, index=False)
except ModuleNotFoundError:
    tvmn_nuvm_ad_cvm = read_csv(saved_diag_path)

show('TVMN and NUVM benchmark KS/AD/CVM diagnostics', tvmn_nuvm_ad_cvm)

show('Existing NUVM goodness-of-fit table', read_csv(NUVM_OUT / '05_NUVM_goodness_of_fit.csv'))

## TVMN and NUVM benchmark KS/AD/CVM diagnostics

,dataset,n,model,KS,AD,CVM
0,diabetes_target,442,TVMN,0.096166,6.866234,1.059989
1,diabetes_target,442,NUVM,0.096039,6.868401,1.057951
2,breast_cancer_mean_radius,569,TVMN,0.071900,8.165759,0.796460
3,breast_cancer_mean_radius,569,NUVM,0.066383,8.020869,0.835427
4,breast_cancer_mean_texture,569,TVMN,0.037387,1.852992,0.203834
5,breast_cancer_mean_texture,569,NUVM,0.038416,1.892433,0.214902
6,breast_cancer_mean_area,569,TVMN,0.124515,17.204953,1.826400
7,breast_cancer_mean_area,569,NUVM,0.125356,16.976849,1.913188
8,wine_alcohol,178,TVMN,0.068165,1.051289,0.176352
9,wine_alcohol,178,NUVM,0.068541,1.056056,0.177197


## Existing NUVM goodness-of-fit table

,Model,KS,AD,CvM
0,Normal,0.079358,35.787213,5.851719
1,Student-t,0.015669,1.091292,0.100901
2,NUVM,0.095112,45.551138,6.503297


# Bootstrap Confidence Interval Tables

In [6]:
show('TVMN bootstrap confidence intervals', read_csv(TVMN_OUT / '04_TVMN_bootstrap_confidence_intervals.csv'))
show('NUVM bootstrap confidence intervals', read_csv(NUVM_OUT / '05_NUVM_bootstrap_confidence_intervals.csv'))
show('NUVM Wald confidence intervals', read_csv(NUVM_OUT / '05_NUVM_wald_confidence_intervals.csv'))
show('TVMN Fisher standard errors and Wald intervals', read_csv(TVMN_OUT / '04_TVMN_fisher_standard_errors_ci.csv'))

## TVMN bootstrap confidence intervals

,parameter,base_estimate,bootstrap_mean,bootstrap_sd,ci_lower_percentile,ci_upper_percentile,successful_bootstrap_replications
0,a,0.574807,0.509218,0.424444,0.000010,1.266165,1000
1,m,0.595383,0.982348,0.516543,0.174861,1.960759,1000
2,b,2.476530,2.158615,0.731302,1.102568,3.629447,1000


## NUVM bootstrap confidence intervals

,parameter,bootstrap_successes,B,mean,std_error,ci_lower,ci_upper
0,mu,1000,1000,0.060261,0.126656,-0.227785,0.357024
1,sigma,1000,1000,1.177823,0.191602,0.933923,1.615385
2,a,1000,1000,0.904650,0.177536,0.337135,0.999000


## NUVM Wald confidence intervals

,parameter,estimate,std_error,ci_lower,ci_upper
0,mu,0.108308,0.000008,0.108293,0.108323
1,sigma,1.246890,0.000010,1.246871,1.246910
2,a,0.999000,0.000008,0.998985,0.999015


## TVMN Fisher standard errors and Wald intervals

,parameter,estimate,std_error,ci_lower,ci_upper,true_value
0,a,0.655566,0.621573,-0.562716,1.873849,0.5
1,m,0.662210,0.623840,-0.560516,1.884936,1.0
2,b,2.627160,0.856650,0.948126,4.306194,2.0


# Simulation Results Tables

In [7]:
show('TVMN Monte Carlo simulation summary', read_csv(TVMN_OUT / '04_TVMN_monte_carlo_summary.csv'))
show('NUVM simulation summary', read_csv(NUVM_OUT / '05_NUVM_simulation_summary.csv'))

## TVMN Monte Carlo simulation summary

,n,replications,successful,a_mean,a_bias,a_mse,a_rmse,m_mean,m_bias,m_mse,m_rmse,b_mean,b_bias,b_mse,b_rmse
0,30,1000,1000,0.725129,0.225129,0.346215,0.588400,0.946031,-0.053969,0.345625,0.587899,1.862184,-0.137816,1.191623,1.091615
1,50,1000,1000,0.703370,0.203370,0.300413,0.548099,0.978167,-0.021833,0.295032,0.543168,1.809209,-0.190791,0.974133,0.986982
2,100,1000,1000,0.675619,0.175619,0.260041,0.509942,1.045281,0.045281,0.250031,0.500031,1.796585,-0.203415,0.658388,0.811411
3,200,1000,1000,0.635867,0.135867,0.219800,0.468828,1.084702,0.084702,0.203176,0.450751,1.788420,-0.211580,0.469260,0.685026
4,500,1000,1000,0.576541,0.076541,0.150615,0.388091,1.090839,0.090839,0.179631,0.423829,1.828277,-0.171723,0.300888,0.548533


## NUVM simulation summary

,n,Bias(mu),RMSE(mu),Bias(sigma),RMSE(sigma),Bias(a),RMSE(a),Convergence Rate,Successful Fits
0,50,0.005116,0.143470,-0.019683,0.106140,-0.124761,0.443107,1.0,1000
1,100,-0.000745,0.099972,-0.008454,0.074725,-0.091587,0.405476,1.0,1000
2,250,-0.000884,0.062104,-0.003936,0.048707,-0.079344,0.338680,1.0,1000
3,500,-0.000073,0.044750,-0.001324,0.032078,-0.074606,0.291750,1.0,1000
4,1000,-0.000663,0.030914,-0.000943,0.023404,-0.043027,0.215077,1.0,1000


# MLE Recovery Tables

In [8]:
show('TVMN MLE recovery table', read_csv(TVMN_OUT / '03_TVMN_MLE_recovery.csv'))
show('NUVM alpha recovery summary', read_csv(NUVM_OUT / 'NUVM_a_recovery_summary.csv'))
show('NUVM method-of-moments / NIFTY table', read_csv(NUVM_OUT / 'NUVM_method_of_moments_NIFTY.csv'))

## TVMN MLE recovery table

,n,true_a,true_m,true_b,a_hat,m_hat,b_hat,m_minus_a,b_minus_m,bias_a,bias_m,bias_b,loglik_hat,loglik_true,loglik_gain,nll,near_degenerate,success,message
0,100,0.5,1.0,2.0,1.144513,1.163456,1.172107,0.018943,0.008651,0.644513,0.163456,-0.827893,-149.347079,-149.467859,0.120780,149.347079,True,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...
1,200,0.5,1.0,2.0,1.065958,1.105019,1.110541,0.039061,0.005523,0.565958,0.105019,-0.889459,-292.740158,-293.668988,0.928830,292.740158,True,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...
2,500,0.5,1.0,2.0,0.804522,0.839295,2.002536,0.034773,1.163241,0.304522,-0.160705,0.002536,-758.008746,-758.267240,0.258495,758.008746,True,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...


## NUVM alpha recovery summary

,True a,Mean MLE a_hat,RMSE MLE a_hat,Mean MoM a_hat,RMSE MoM a_hat,Convergence Rate
0,0.2,0.233910,0.260186,0.218977,0.245733,1.0
1,0.5,0.424813,0.289351,0.422108,0.305494,1.0
2,0.8,0.760735,0.181221,0.726293,0.225277,1.0


## NUVM method-of-moments / NIFTY table

,mu_mom,sigma_mom,a_mom_raw,a_mom,sample_kurtosis,boundary_flag
0,0.042126,1.054237,4.513236,0.999,23.369296,True


# Paper-Ready Interpretation Snippets

In [9]:
for path in [
    TVMN_OUT / '05_TVMN_benchmark_interpretation.md',
    TVMN_OUT / '05_TVMN_good_result_summary.md',
    TVMN_DIR / 'TVMN_motivation_section.md',
    TVMN_DIR / 'TVMN_NUVM_reviewer_audit_checklist.md',
]:
    if path.exists():
        display(Markdown(path.read_text(encoding='utf-8')))
    else:
        print('Missing:', path)

# TVMN Strong Benchmark Evidence

This section compares TVMN and NUVM against Normal, Laplace, Student-t, Gamma, Lognormal, Weibull, and Exponential models on raw positive datasets.

## AIC Winners

| Dataset | n | AIC winner | TVMN AIC - best AIC |
|---|---:|---|---:|
| diabetes_target | 442 | Gamma | 60.240 |
| breast_cancer_mean_radius | 569 | Lognormal | 75.996 |
| breast_cancer_mean_texture | 569 | Lognormal | 37.914 |
| breast_cancer_mean_area | 569 | Lognormal | 189.468 |
| wine_alcohol | 178 | Normal | 4.001 |
| wine_malic_acid | 178 | Lognormal | 61.396 |
| wine_proline | 178 | Lognormal | 40.028 |
| linnerud_weight | 20 | Laplace | 3.859 |

TVMN AIC wins: 0/8.

## TVMN Versus NUVM

TVMN has higher log-likelihood than NUVM in 5/8 benchmark datasets.

TVMN has a smaller KS statistic than NUVM in 6/8 benchmark datasets.

The detailed comparison is saved in `05_TVMN_benchmark_TVMN_vs_NUVM.csv`.

## Interpretation

These expanded comparisons are the main reviewer-facing evidence. If TVMN wins or nearly ties on some datasets, the manuscript can claim competitive empirical performance. If it does not win, the paper should emphasize mathematical novelty, valid estimation, and situations where variance-mixture flexibility is practically useful.

## Recommended Wording

> Across several benchmark datasets, TVMN was compared with common symmetric, heavy-tailed, and positive-support distributions using log-likelihood and information criteria. The results show where the additional triangular variance-mixture structure is empirically useful and where simpler alternatives remain preferable.


# TVMN Good Result Summary

The expanded benchmark does not support a claim that TVMN dominates all competing distributions by AIC. However, it does provide useful positive results that can be reported honestly.

## Best Defensible Positive Result

TVMN improves over NUVM in several direct comparisons:

- Higher log-likelihood than NUVM in 5 out of 8 benchmark datasets.
- Smaller KS statistic than NUVM in 6 out of 8 benchmark datasets.
- Better AIC and BIC than NUVM for the breast cancer mean area dataset.

The strongest TVMN-versus-NUVM case is:

| Dataset | TVMN logLik - NUVM logLik | TVMN AIC - NUVM AIC | TVMN BIC - NUVM BIC |
|---|---:|---:|---:|
| breast_cancer_mean_area | 8.139 | -14.278 | -9.934 |

## Careful Manuscript Wording

> In the expanded benchmark study, TVMN did not uniformly dominate all classical competitors by AIC. However, relative to NUVM, TVMN produced higher likelihoods in five of eight datasets and smaller KS statistics in six of eight datasets. For the breast cancer mean area data, TVMN improved both AIC and BIC relative to NUVM, indicating that the triangular variance-mixture structure can provide practical gains over the uniform variance-mixture model in some settings.


# Motivation for the TVMN/NUVM Distribution Paper

Many empirical datasets depart from the classical Normal model because they show heavier tails, changing local variability, or uncertainty in the dispersion structure. A fixed-variance Normal distribution is analytically convenient, but it can be restrictive when the spread of the data varies across latent regimes.

Variance-mixture models address this limitation by keeping a Normal conditional kernel while treating the variance as a positive random variable. This preserves interpretability and computational convenience, while allowing the marginal distribution to express more flexible shape and tail behavior.

The NUVM model uses a uniform mixing law for the variance. This allows the variance to fluctuate over an interval, but it gives equal weight to all variance values in that interval. The TVMN model extends this idea by using a triangular variance distribution with lower bound, mode, and upper bound. The mode parameter lets the model place more mass near smaller, central, or larger variance values.

Practically, TVMN should be used as an additional flexible benchmark model rather than as a universal replacement for standard distributions. It is most relevant when Normal tails are too light, Laplace alternatives are too sharply peaked, or the analyst wants a bounded and interpretable representation of latent variance uncertainty.

The paper's main contribution is threefold: it defines the TVMN distribution and its distributional functions; it develops likelihood-based estimation with simulation, Fisher information, and bootstrap evidence; and it evaluates the model against established competitors using real-data benchmarks, information criteria, and goodness-of-fit diagnostics.


# TVMN/NUVM Reviewer-Readiness Audit Checklist

## Theory

- [x] Definition of distribution.
- [x] PDF.
- [x] CDF.
- [x] Survival function.
- [x] Hazard function.
- [x] Reverse hazard function.
- [x] Cumulative hazard function.
- [ ] Closed-form quantile function.
- [x] Numerical quantile computation can be added through CDF inversion.
- [x] Moments.
- [x] Kurtosis.
- [x] Numerical verification that the PDF integrates to one.

## Estimation

- [x] MLE likelihood.
- [x] Log-likelihood.
- [x] Numerical optimization procedure.
- [x] Parameter constraints enforced by transformation.
- [x] Fisher information and Wald intervals.
- [x] Bootstrap confidence intervals.

## Simulation

- [x] Multiple sample sizes.
- [x] Bias.
- [x] MSE.
- [x] RMSE.
- [x] Convergence summaries.

## Real Data

- [x] Multiple datasets.
- [x] Competing distributions.
- [x] Log-likelihood.
- [x] AIC/BIC/CAIC/HQIC.
- [x] KS diagnostics.
- [x] AD/CVM diagnostics computed in `TVMN_NUVM_submission_ready_results.ipynb`.
- [x] Histogram with fitted density.
- [x] Empirical CDF comparison.
- [x] Hazard shape plots.

## Current Reviewer Position

The strongest position is that TVMN is mathematically valid, estimable, and competitive with NUVM in selected datasets. The current evidence does not support a universal superiority claim over all classical alternatives.


# Export Notes

For manuscript writing, use the displayed tables directly instead of inventing values. If new experiments are run, rerun this notebook so every table reflects the current CSV outputs.